In [ ]:
import sys

sys.path.append("../src")

from sqlalchemy import and_, desc, func, or_, select
from sqlalchemy.orm import Session, joinedload, selectinload

from domain import (
    engine,
    User,
    Category,
    Ingredient,
    Recipe,
    Rating,
    Favorite,
    Follow,
    RecipeCategory,
    RecipeIngredient,
    RecipeStep,
 )

In [ ]:
with Session(engine) as session:
    stmt = select(Recipe).order_by(Recipe.title)
    recipes = session.scalars(stmt).all()

    for recipe in recipes:
        print(recipe.recipe_id, recipe.title, recipe.servings)

In [ ]:
with Session(engine) as session:
    stmt = (
        select(Recipe.title, func.avg(Rating.stars).label("avg_stars"), func.count(Rating.rating_id).label("num_ratings"))
        .outerjoin(Rating, Rating.recipe_id == Recipe.recipe_id)
        .group_by(Recipe.recipe_id, Recipe.title)
        .order_by(desc("avg_stars"), Recipe.title)
    )

    for title, avg_stars, num_ratings in session.execute(stmt):
        print(title, "| avg:", avg_stars, "| ratings:", num_ratings)

In [ ]:
with Session(engine) as session:
    stmt = (
        select(User.username, func.count(Follow.followed_id).label("followers"))
        .outerjoin(Follow, Follow.followed_id == User.user_id)
        .group_by(User.user_id, User.username)
        .order_by(desc("followers"), User.username)
    )

    for username, followers in session.execute(stmt):
        print(username, followers)